# CLIP embeddings for the one-shot separability experiments

Computes CLIP image embeddings (and text embeddings of the class names) for **CIFAR-100** (downloads automatically) or **ImageNet-val** (local folder) and saves one `.npz` file.

Run the cells top to bottom. Parameters are in the second code cell. On a GPU this takes about a minute for CIFAR-100; on a CPU 10–15 minutes.

Output `.npz` keys: `X` (raw image embeddings, `[N, d]`), `y` (class index, `[N]`), `T` (text embeddings of the class prompts, `[C, d]`), `classes` (class names).

In [ ]:
# Install dependencies (run once; restart the kernel afterwards if torch was not installed before)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "open_clip_torch", "numpy", "tqdm"], check=False)

In [ ]:
# ---------------- parameters ----------------
DATASET    = "cifar100"        # "cifar100" or "imagenet"
ROOT       = "./data"          # download folder for cifar100; for imagenet: the val folder in ImageFolder layout <root>/<wnid>/*.JPEG
OUT        = "embeddings_cifar100.npz"
MODEL      = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"
BATCH      = 256
MAX_IMAGES = 0                 # 0 = all; e.g. 20000 to subsample
COMPUTE_TEXT = True
WORKERS    = 0                 # keep 0 in Jupyter (multiprocessing workers can hang in notebooks)

In [ ]:
import numpy as np, torch, torchvision, open_clip
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

model, _, preprocess = open_clip.create_model_and_transforms(MODEL, pretrained=PRETRAINED)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer(MODEL)

In [ ]:
# ---------------- dataset ----------------
if DATASET == "cifar100":
    ds = torchvision.datasets.CIFAR100(root=ROOT, train=False, download=True, transform=preprocess)
    classes = list(ds.classes)
    targets = np.asarray(ds.targets)
else:
    ds = torchvision.datasets.ImageFolder(root=ROOT, transform=preprocess)
    classes = list(ds.classes)
    targets = np.asarray(ds.targets)
    try:
        from open_clip import IMAGENET_CLASSNAMES
        if len(IMAGENET_CLASSNAMES) == len(classes):
            classes = list(IMAGENET_CLASSNAMES)
    except Exception:
        pass

if MAX_IMAGES and MAX_IMAGES < len(ds):
    rng = np.random.default_rng(0)
    idx = np.sort(rng.choice(len(ds), size=MAX_IMAGES, replace=False))
    ds = Subset(ds, idx.tolist())
    targets = targets[idx]
print(f"{DATASET}: {len(ds)} images, {len(classes)} classes")

loader = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=(device == "cuda"))

In [ ]:
# ---------------- image embeddings ----------------
feats = []
with torch.no_grad():
    for images, _ in tqdm(loader, desc="images"):
        feats.append(model.encode_image(images.to(device)).float().cpu().numpy())
X = np.concatenate(feats, axis=0).astype(np.float32)
y = targets.astype(np.int64)
print("X", X.shape, "y", y.shape)

In [ ]:
# ---------------- text embeddings ----------------
T = None
if COMPUTE_TEXT:
    prompts = [f"a photo of a {c.replace('_', ' ')}." for c in classes]
    with torch.no_grad():
        toks = tokenizer(prompts).to(device)
        T = np.concatenate([model.encode_text(toks[i:i+256]).float().cpu().numpy()
                            for i in range(0, len(prompts), 256)], axis=0).astype(np.float32)
    print("T", T.shape)

In [ ]:
# ---------------- save ----------------
out = dict(X=X, y=y, classes=np.array(classes), model=MODEL, pretrained=PRETRAINED, dataset=DATASET)
if T is not None:
    out["T"] = T
np.savez_compressed(OUT, **out)
print("saved", OUT, f"({X.nbytes/1e6:.0f} MB of embeddings)")

# quick sanity check: zero-shot accuracy with the text embeddings (should be ~60-70% on CIFAR-100 for ViT-B/32)
if T is not None:
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True); Tn = T / np.linalg.norm(T, axis=1, keepdims=True)
    acc = (np.argmax(Xn @ Tn.T, axis=1) == y).mean()
    print(f"zero-shot accuracy sanity check: {acc:.3f}")